# Gemini API

* * * 

<div class="alert alert-success">  
    
### Learning Objectives 
    
1. **Call the Gemini API** with structured schemas for consistent classifications
2. **Process multiple texts** efficiently with batch classification
3. **Validate your results** using three practical approaches:
   - Spot-checking: Random inspection of outputs
   - Consistency testing: Running same text multiple times
   - Comparative validation: Comparing to your own coding
4. **Recognize red flags** in LLM output (generic reasoning, hallucination, contradictions)
5. **Know when NOT to use LLMs** for classification tasks

</div>

* * *


<a id='intro'></a>

# Introduction to Structured Outputs

## Why Structured Outputs Matter for Social Science Research

When using LLMs to classify or annotate social science data, we face a critical challenge: **reliability and consistency**. Traditional prompt-based approaches can return varied formats, requiring extensive post-processing and validation. This introduces:

1. **Data quality issues**: Inconsistent formatting makes downstream analysis difficult
2. **Reproducibility concerns**: Different runs may produce differently structured outputs
3. **Validation complexity**: Checking if the model followed instructions requires manual review

**Structured outputs** solve these problems by **guaranteeing** that Gemini returns data in a predefined schema. This is essential for:

- **Content classification**: Sentiment analysis, topic coding, moderation decisions
- **Data extraction**: Pulling specific fields from unstructured text (dates, names, claims)
- **Multi-dimensional coding**: Assigning multiple labels simultaneously (e.g., emotion + intensity + justification)
- **Reliability**: Every response matches your schema exactly

## LLM-as-Judge Paradigm

In computational social science, the **LLM-as-judge** approach uses language models to replicate human annotation tasks. For example:
- Coding political speech for policy positions
- Classifying social media content by toxicity
- Judging moral dilemmas (like r/AITA posts)
- Identifying misinformation or harmful content

Structured outputs ensure that your LLM judge provides consistent, machine-readable judgments that can be directly integrated into quantitative analyses.

<a id='setup'></a>

# Setting Up Your Environment

## Getting API Keys

 **Warning:** You currently cannot use Google Gemini with your Berkeley account, so you will have to use your personal account.

1. Go to [Google AI Studio](https://aistudio.google.com/)
2. Sign in with your Google account
3. Click "Get API Key" or "Create API Key"
4. You may need to create a Project on the [Google Cloud Console](https://console.cloud.google.com/)
5. **Important:** Copy this key and save it somewhere safe. Don't share it publicly!

## Free Tier for Research

The Gemini API offers a **generous free tier** perfect for research:
- **Gemini 2.5 Flash**: Up to 15 requests per minute (RPM)
- **Gemini 2.5 Pro**: Up to 2 RPM
- **Daily token limits**: Sufficient for moderate-scale research projects

 **Billing Advisory**: Do not add a billing account unless you explicitly need to exceed free tier limits. The API will simply fail with a quota error rather than charging you.

## Installation

We need two key packages:
1. **google-genai**: The official Google Generative AI SDK (newer, simpler API)
2. **pydantic**: For defining structured schemas

 Note: If you're on DataHub, uncomment and run the installation cell below, then **restart your kernel**.

In [ ]:
# Uncomment to install (restart kernel after running)
# !pip install -U -q google-genai pydantic

In [1]:
import os
import json
import pandas as pd
from typing import List, Optional, Literal
from pydantic import BaseModel, Field
from google import genai

## Setting Up API Key

In real projects, we never hard-code API keys in notebooks.
Instead, we store them in a hidden file called .env and load them securely in Python.

This keeps keys:
- out of your code
- out of GitHub
- safe to share the notebook
- easy to swap between machines

STEPS
1. Create a .env file (in your project folder)
2. In the root of your project, create a file named .env and type: `GOOGLE_API_KEY="your_key_here"`
3. Be sure your .gitignore includes `.env`! This prevents accidental commits.

In [2]:
#%conda install python-dotenv

In [2]:
import os
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Read the key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if GOOGLE_API_KEY is None:
    raise ValueError("GOOGLE_API_KEY not found. Did you create a .env file?")

# Initialize the client
client = genai.Client(api_key=GOOGLE_API_KEY)

# Basic Classification

Let's start with a simple classification task -- sentiment.

In [ ]:
# Simple, unstructured prompt
text = """This product completely changed my life! I've been using it for 3 months 
and can't imagine going back. Highly recommend to anyone looking for a solution."""

prompt = f"""Classify the sentiment of the following text as positive, neutral, or negative.
Also provide a brief explanation for your classification.

Text: {text}

Classification:"""

for i in range(3):
    # Call the API without any structure
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt)
    print(response.text)
    print('\n')


Classification: Positive

Explanation: The user expresses strong satisfaction with the product, using phrases like "completely changed my life!", "can't imagine going back," and "Highly recommend." These are all strong indicators of a positive experience.


Classification: **Positive**

Explanation: The text uses strong positive language and endorsements, such as "completely changed my life!", "can't imagine going back," and "Highly recommend." These phrases convey enthusiasm and a very satisfying experience with the product.




### The Problem with Unstructured Outputs

Notice several issues:

**1. Inconsistent Format**
- Sometimes: "Classification: Positive"
- Sometimes: "Classification: **Positive**..."

**2. Hard to Parse Programmatically**
- How do you extract just "positive" reliably?
- Need fragile regex or string matching
- "Positive" vs "positive" vs "POSITIVE"?

**3. Not Analysis-Ready**
- Can't directly put this in a pandas DataFrame
- Can't count categories or aggregate results
- Requires manual post-processing for each text

**4. No Validation**
- Model could say "very positive" (not in your categories)
- Could give a paragraph instead of a simple label
- No guarantee you'll get what you asked for

**For 10 texts, this is annoying. For 10,000 texts, it's a disaster.**

This is why we use structured outputs...


<a id='basic'></a>

# Structured Classification

Some LLMs, like Gemini, allow us to classify using a structured schema.

## Step 1: Define Your Schema with Pydantic

Pydantic models define the **exact structure** of the data you want back from Gemini. Think of it as creating a template for your annotations.

In [15]:
class SentimentClassification(BaseModel):
    """Schema for basic sentiment classification."""
    
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        description="The overall sentiment of the text"
    )
    reasoning: str = Field(
        description="Brief explanation (1-2 sentences) for the classification",
        max_length=200
    )

## Step 2: Make a Structured Request

Key parameters:
- `model`: We'll use `gemini-2.5-flash` (fast, free tier)
- `response_mime_type`: Must be `"application/json"`
- `response_json_schema`: The Pydantic schema converted to JSON Schema

What happens:

- Schema becomes part of the prompt - The API injects the schema into the system prompt/context, telling the model "you must respond in this exact JSON format"
- Server-side validation - After the model generates output, Google's API validates it against the schema before sending it back to you. If it doesn't match, they'll either retry (sometimes transparently), or return an error
- Constrained decoding (possibly) - Some systems use the schema to actually constrain token generation in real-time, making it impossible for the model to generate invalid JSON. It's not entirely clear if Gemini does this, but many structured output systems do.



In [29]:
# Example text to classify
text = """This product completely changed my life! I've been using it for 3 months 
and can't imagine going back. Highly recommend to anyone looking for a solution."""

# Create the prompt
prompt = f"""Classify the sentiment of the following text:

{text}
"""

# Make the request with structured output
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config={
        # Specify that we want a structured JSON response
        "response_mime_type": "application/json",
        "response_json_schema": SentimentClassification.model_json_schema(),
    },
)

# Parse the response directly into our Pydantic model
result = SentimentClassification.model_validate_json(response.text)

In [30]:
print(result.model_dump_json(indent=2))

{
  "sentiment": "positive",
  "reasoning": "The user expresses strong satisfaction, stating the product \"completely changed my life\" and \"highly recommends\" it, indicating a very positive experience."
}


##  Question: What Are the Advantages?

Compare the structured output above to traditional prompting:

**Traditional Prompting:**
```
"The sentiment is positive because the user expresses satisfaction..."
```
→ Requires parsing, might vary in format, hard to validate

**Structured Output:**
```json
{
  "sentiment": "positive",
  "confidence": "high",
  "reasoning": "..."
}
```
This is a guaranteed format, type-safe, directly usable in analysis

<a id='aita'></a>

# LLM-as-Judge: AITA Classification

Now let's apply structured outputs to a real computational social science use case: classifying moral judgments in Reddit's "Am I The Asshole" (r/AITA) community.

## Research Context

r/AITA posts present moral dilemmas where users seek judgment on their behavior. The community uses specific labels:
- **NTA** (Not The Asshole): The poster is justified
- **YTA** (You're The Asshole): The poster is at fault
- **ESH** (Everyone Sucks Here): All parties are at fault
- **NAH** (No Assholes Here): No one is at fault
- **INFO** (Need More Info): Cannot determine without additional context

## Defining a Comprehensive AITA Schema

For research, we want more than just the verdict. We want to capture multiple dimensions of the moral judgment:

In [100]:
class AITAJudgment(BaseModel):
    """Structured schema for AITA moral judgment classification."""
    
    verdict: Literal["YTA", "NTA", "ESH", "NAH", "INFO"] = Field(
        description="Primary judgment: YTA (You're the Asshole), NTA (Not the Asshole), "
                   "ESH (Everyone Sucks Here), NAH (No Assholes Here), INFO (Need More Info)"
    )
    
    moral_domain: List[Literal["harm", "fairness", "loyalty", "authority", "sanctity"]] = Field(
        description="Relevant moral foundations from Moral Foundations Theory",
        min_length=1,
        max_length=3
    )
    
    severity: Literal["minor", "moderate", "major"] = Field(
        description="How serious is the moral violation (if any)"
    )
    
    justification: str = Field(
        description="Detailed explanation for the judgment (2-3 sentences)",
        min_length=50,
        # set this to a higher value to allow for more detailed reasoning
        max_length=700
    )
    
    key_factors: List[str] = Field(
        description="2-4 specific factors that influenced the judgment",
        min_length=2,
        max_length=4
    )


## Example: Classifying an AITA Post

Let's test our schema on a sample post:

In [101]:
# Sample AITA post
aita_post = """
I (28F) have been with my partner (30M) for 5 years. Last month, he got a promotion 
that requires him to relocate to another state. He accepted without discussing it 
with me first, just told me we'd be moving in 3 months. When I said I couldn't leave 
my job and my family, he said I was being selfish and not supporting his career. 
I told him if he moves, we're done. Now he's saying I'm giving him an ultimatum 
and that I never cared about his success. AITA for not wanting to move?
"""

# Create research-focused prompt
prompt = f"""Analyze the following AITA post and provide a judgment.
Consider multiple moral dimensions and be specific in your reasoning.

Post:
{aita_post}

Provide your analysis following the structured format.
"""

# Get structured judgment
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": AITAJudgment.model_json_schema(),
    },
)

judgment = AITAJudgment.model_validate_json(response.text)

In [102]:
print(judgment.model_dump_json(indent=2))

{
  "verdict": "NTA",
  "moral_domain": [
    "fairness",
    "loyalty",
    "harm"
  ],
  "severity": "major",
  "justification": "The partner made a significant, life-altering decision (relocating for a promotion) unilaterally without any prior discussion with his long-term partner. This demonstrates a profound lack of respect and partnership, completely disregarding her career, family ties, and autonomy. His subsequent accusations of selfishness and lack of support, when she expressed valid concerns about uprooting her life, are manipulative and further highlight his self-centered approach to their shared future. The OP is not the asshole for refusing to abandon her established life for a decision made without her input.",
  "key_factors": [
    "Partner's unilateral decision to accept relocation",
    "Partner's disregard for OP's job, family, and input",
    "Partner's accusation of OP being selfish and unsupportive",
    "The significant life changes demanded of the OP without he

##  Reflection: Comparing Judgments

Let's see how the model's judgment changes when we modify the post slightly:

In [103]:
# Modified version with additional context
aita_post_v2 = """
I (28F) have been with my partner (30M) for 5 years. Last month, he got a promotion 
that requires him to relocate to another state. He accepted without discussing it 
with me first, just told me we'd be moving in 3 months. When I said I couldn't leave 
my job - I'm a senior nurse at a children's hospital and my team depends on me - and 
my elderly mother who I'm the primary caregiver for, he said I was being selfish. 
I told him if he moves, we're done. He's now telling our friends I gave him an unfair 
ultimatum. AITA for not wanting to move?
"""

prompt_v2 = f"""As a neutral observer, analyze the following AITA post and provide a structured judgment.
Consider multiple moral dimensions and be specific in your reasoning.

Post:
{aita_post_v2}

Provide your analysis following the structured format.
"""

response_v2 = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_v2,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": AITAJudgment.model_json_schema(),
    },
)

judgment_v2 = AITAJudgment.model_validate_json(response_v2.text)

In [104]:
print(judgment_v2.model_dump_json(indent=2))

{
  "verdict": "NTA",
  "moral_domain": [
    "fairness",
    "loyalty"
  ],
  "severity": "major",
  "justification": "Your partner made a unilateral decision about a significant life change, disregarding your established career, caregiving responsibilities, and the nature of your five-year relationship. His attempt to frame you as selfish for having legitimate reasons not to uproot your life, and then complaining to friends, demonstrates a profound lack of respect and partnership. You are not obligated to follow his non-negotiated decisions, and your 'ultimatum' is a boundary in response to his disrespectful actions.",
  "key_factors": [
    "Partner's unilateral decision-making on a major life event",
    "Partner's dismissal of OP's professional and caregiving responsibilities",
    "Partner calling OP 'selfish' and complaining to friends",
    "OP's valid reasons for not wanting to relocate"
  ]
}


Notice how Gemini added a moral domain (loyalty) in the second classification, and used different key factors. 

Even with structured outputs guaranteeing the format, the **content** can still vary between runs due to its stochastic nature. We'll learn how to check consistency later in the notebook.

<a id='batch'></a>

# Batch Processing for Research

In real research, you'll need to classify many posts. Here's how to do batch processing efficiently:

## Creating Sample Data

Let's create a small dataset of AITA posts for demonstration:

In [105]:
%pwd

'/Users/tomvannuenen/Library/CloudStorage/Dropbox/GitHub/DEV/COMPSS-211/lessons/week11_cloud-computing'

In [125]:
df = pd.read_csv("../../data/aita_top_subs.csv")
df = df[df['flair_css_class'].isin(['ass', 'not'])]
df = df[~df['selftext'].isin(['[removed]', '[deleted]'])]

df['reddit_verdict'] = df['flair_css_class'].replace({
    'ass': 'YTA',
    'not': 'NTA'
})

In [126]:
df_sample = df.groupby('flair_css_class').sample(n=2, random_state=42).reset_index(drop=True)
df_sample

,idint,idstr,created,self,nsfw,author,title,url,selftext,score,...,flair_text,flair_css_class,augmented_at,augmented_count,created_date,year,month,day_of_week,text_length,reddit_verdict
0,1646409815,t3_r889xz,1638563173,1.0,0.0,RelationshipTotal375,AITA for refusing to spend Christmas with my f...,NaN,My mom and my wife avoid each other if they ca...,3997.0,...,Asshole,ass,NaN,NaN,2021-12-03 20:26:13,2021,12,Friday,2133.0,YTA
1,808226476,t3_dd73jg,1570194309,1.0,0.0,neatoSarah,AITA for masturbating alone in my dorm room?,NaN,"Weird one here. So I live in a triple dorm, I ...",5630.0,...,Everyone Sucks,ass,NaN,NaN,2019-10-04 13:05:09,2019,10,Friday,1803.0,YTA
2,1234355296,t3_kewin4,1608207923,1.0,0.0,partyontheclouds,AITA for refusing to pay for a child's surgica...,NaN,"On mobile, apologies for the bad formatting. \...",15038.0,...,Not the A-hole,not,NaN,NaN,2020-12-17 12:25:23,2020,12,Thursday,1621.0,NTA
3,753157644,t3_cges5o,1563809990,1.0,0.0,whamanraman,WIBTA if I take my coworker to HR for touching...,NaN,"Okay, so here’s the story.\n\n \nWhen my old c...",10521.0,...,Not the A-hole,not,NaN,NaN,2019-07-22 15:39:50,2019,7,Monday,2041.0,NTA


## Batch Classification Function

Let's create a function that handles rate limiting and error handling:

In [127]:
def classify_aita_batch(posts: pd.DataFrame, 
                        text_col: str = 'selftext',
                        id_col: str = 'idstr',
                        delay: float = 4.0) -> pd.DataFrame:
    """
    Classify a batch of AITA posts with rate limiting.
    
    Args:
        posts: DataFrame with posts to classify
        text_col: Name of column containing text (default 'selftext')
        id_col: Name of column containing post IDs (default 'idstr')
        delay: Seconds to wait between requests (default 4.0 for free tier)
    
    Returns:
        DataFrame with original posts and judgments
    """
    results = []
    
    for i, (_, row) in enumerate(posts.iterrows()):
        print(f"\n[{i+1}/{len(posts)}] Processing post {row[id_col]}...")
        
        try:
            prompt = f"""As a neutral observer, analyze the following AITA post and provide a structured judgment.

Post:
{row[text_col]}

Provide your complete analysis with verdict, moral domains, severity, justification, and key factors."""
                                
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_json_schema": AITAJudgment.model_json_schema(),
                },
            )
            
            judgment = AITAJudgment.model_validate_json(response.text)
            
            result = {
                "idstr": row[id_col],
                "selftext": row[text_col],
                **judgment.model_dump()
            }
            results.append(result)
            print(f"Verdict: {judgment.verdict}")
            
        except Exception as e:
            print(f"Error: {e}")
            results.append({
                "idstr": row[id_col],
                "selftext": row[text_col],
                "error": str(e)
            })
        
        if i < len(posts) - 1:
            time.sleep(delay)
    
    return pd.DataFrame(results)

## Run Batch Classification

 **Note**: This will take about 12 seconds due to rate limiting (4 seconds between requests).

In [128]:
# Run batch classification
results_df = classify_aita_batch(df_sample, text_col='selftext')


[1/4] Processing post t3_r889xz...
Verdict: YTA

[2/4] Processing post t3_dd73jg...
Verdict: YTA

[3/4] Processing post t3_kewin4...
Verdict: NTA

[4/4] Processing post t3_cges5o...
Verdict: NTA


In [136]:
# Merge back with original data to keep Reddit's verdict
results_df = results_df.merge(
    df_sample[['idstr', 'reddit_verdict']], 
    on='idstr'
)
results_df

,idstr,selftext,verdict,moral_domain,severity,justification,key_factors,reddit_verdict
0,t3_r889xz,My mom and my wife avoid each other if they ca...,YTA,"[harm, fairness, loyalty]",moderate,You are the asshole for overreacting to a mino...,"[Overreaction to a non-malicious interaction, ...",YTA
1,t3_dd73jg,"Weird one here. So I live in a triple dorm, I ...",YTA,"[harm, fairness]",major,OP's actions are a deliberate and calculated a...,[Intentional harassment plan to drive out room...,YTA
2,t3_kewin4,"On mobile, apologies for the bad formatting. \...",NTA,"[fairness, harm, authority]",moderate,You were legally exonerated for the accident a...,"[OP's legal exoneration from fault, Substantia...",NTA
3,t3_cges5o,"Okay, so here’s the story.\n\n \nWhen my old c...",NTA,"[harm, fairness]",moderate,The OP is absolutely not the asshole; Louis's ...,[Louis's repeated disregard for explicit bound...,NTA


In [130]:
results_df.to_csv("aita_classification_results.csv", index=False)

<a id='validation'></a>

# Validating Your Results

## Why Validation Matters

You've learned how to call the API and get structured classifications. But the most important skill is knowing when to trust the output.

Recent research shows that new analysts using AI tools can be **less effective than without them** because they lack validation skills. Let's learn how to check your work.

## Three Simple Validation Approaches

We'll cover three practical methods you can use in any project:
1. **Spot-checking**: Randomly sample and inspect results
2. **Consistency checking**: Test if you get the same answer twice
3. **Comparative validation**: Compare LLM to your own judgment

Let's implement each one.

## 1. Spot-Checking: Random Inspection

The simplest validation: randomly sample some results and read them carefully.

**What to look for:**
- Does the justification actually reference the text content?
- Is the reasoning generic ("This is positive because it's good") or specific?
- Are there obvious contradictions (e.g., "minor" severity but "YTA" verdict)?
- Does it make sense to you?


In [139]:
import textwrap

# Let's spot-check a few results

for _, row in results_df.iterrows():
    print(f"\n{'='*80}")
    print(f"POST: {row['idstr']}")
    print(f"{'='*80}")
    print(f"TEXT:\n{textwrap.fill(row['selftext'], width=80)}\n")
    print(f"VERDICT: {row['verdict']} ({row['severity']})")
    print(f"MORAL DOMAINS: {', '.join(row['moral_domain'])}")
    print(f"JUSTIFICATION: {textwrap.fill(row['justification'], width=80)}\n")
    print("KEY FACTORS:")
    for factor in row['key_factors']:
        wrapped_factor = textwrap.fill(factor, width=76, initial_indent="  • ", subsequent_indent="    ")
        print(wrapped_factor)


POST: t3_r889xz
TEXT:
My mom and my wife avoid each other if they can. There was never a big incident.
My mom isn't a JNMIL and my wife isn't some horrible DIL, but if they can get
away with not speaking, they don't speak.  For a little while my mom tried to
keep her close relationship with me by trying to see me when my wife wasn't
there, but I felt weird about that, my life got busy, and more and more I told
my mom the only times I was available were when my wife would be there, so my
mom lost interest in seeing me. In the four years I've been married I've seen my
mom a handful of times. It was hard on me for a while, but I feel like what am I
going to do when the two of them just don't want to interact.  My mom did sent a
gift when my 3 year old was born, and she has met her a couple of times. My mom
is always polite when we see her, but obviously there can't be a traditional
grandma role when she doesn't want to come around.  My wife's family lives
across the country, so we spend 

**What to look for when spot-checking:**
1. Does the justification actually reference specific content from the text?
2. Is the reasoning generic ("they were wrong") or specific ("they violated an agreement")?
3. Do the moral domains align with what's actually discussed?
4. Would you have coded it the same way?

Spot-checking is your first line of defense against bad classifications. If something looks off here, investigate further.

## 2. Consistency Checking: Testing Reliability

LLMs have some randomness. A good test: classify the same text multiple times. Do you get the same answer?

**Interpreting results:**
- Same answer every time: High confidence in that classification
- Different answers: This case is ambiguous - flag for review


In [145]:
def check_consistency(text: str, n_trials: int = 5):
    """Classify the same text multiple times and check agreement."""
    
    prompt = f"Analyze this AITA post and provide a structured judgment.\n\n{text}"
    
    verdicts = []
    
    for i in range(n_trials):
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config={
                "response_mime_type": "application/json",
                "response_json_schema": AITAJudgment.model_json_schema(),
                "temperature": 0.7
            }
        )
        result = AITAJudgment.model_validate_json(response.text)
        verdicts.append(result.verdict)
        print(f"Trial {i+1}: {result.verdict}")

In [146]:
# Run on first post from your sample
ambiguous_text = """
My sister (18f) has social anxiety, and she's just a very awkward person overall. 
Yesterday, she asked me to go in the store and buy her a plan b because she was 
too embarrassed.

Also, they had them locked up, so you had to ask someone who works there to unlock 
it for you, and my sister gets anxious going up to and talking to people she doesn't 
know. I told her she had to go in and get it herself.

She got mad at me and said I was being a bad sister because she asked for a simple 
request, and I should've done it, knowing she has social anxiety.

I get where she's coming from, but I also feel like she needs to stop being afraid 
and learn to talk to people, even if she's embarrassed. She also would've done it 
if I wasn't there. AITA?
"""

check_consistency(ambiguous_text, n_trials=5)

Trial 1: YTA
Trial 2: YTA
Trial 3: YTA
Trial 4: YTA
Trial 5: YTA


## 3. Comparative Validation: Checking Against a Baseline

The gold standard: compare LLM classifications to some baseline judgment.

**For AITA posts, we have Reddit's verdict** - the label given by the community. This isn't "ground truth" in an absolute sense (people disagree about moral judgments!), but it's a useful baseline to see if the LLM is making similar judgments to the Reddit community.

**Process:**
1. Take posts where Reddit has already given a verdict
2. Run LLM classification on the same posts
3. Compare: How often does the LLM agree with Reddit?

**For your own data:**
- If you don't have existing labels, you'll need to manually code a sample yourself
- Code 10-20 examples first (don't look at LLM results yet!)
- Then run LLM and compare

**Rule of thumb for agreement rates:**
- 80%+ agreement: LLM aligns well with your baseline
- 60-80% agreement: Proceed with caution, document limitations  
- <60% agreement: Don't trust it - either improve prompt or use human coding

**Important caveat:** High agreement doesn't mean "correct" - it means the LLM is consistent with your baseline. If Reddit's verdicts are biased, the LLM might learn those same biases!

In [ ]:
# how many out of 4 are agreed upon
(results_df['reddit_verdict'] == results_df['verdict']).sum()

4

<a id='when-to-use'></a>

# When to Use (and Not Use) LLMs for Classification

## Good Use Cases

**Exploratory analysis**
- You're getting a sense of your data
- You need to quickly prototype coding schemes
- You're generating hypotheses, not testing them

**Large-scale pattern detection**
- You have thousands of texts
- You've validated on a sample and agreement is >80%
- You'll report your validation metrics

**Augmenting human coding**
- LLM does first pass, humans review edge cases
- You use consistency checks to flag ambiguous cases

## Inappropriate Use Cases

**High-stakes decisions**
- Anything affecting people's lives (medical, legal, hiring)
- Published research without validation
- Policy recommendations

**When you lack domain expertise**
- If you can't judge whether results make sense, you can't supervise effectively
- Get domain knowledge first, then use LLMs

**Very small samples**
- If you only have 30 texts, just code them yourself
- Setup and validation time exceeds benefit

## The Decision Framework

Ask yourself:
1. Can I validate the results against my own judgment?
2. What's the cost of errors? (Exploratory vs. published vs. real-world impact)
3. Is my agreement rate >80% on a validation sample?

# Key Takeaways

1. Use structured outputs (Pydantic schemas) for consistent classifications
2. Handle batch processing with rate limits and delays
3. **Validate your results** - spot-checking, consistency testing, human comparison
4. **Look for red flags** - generic reasoning, hallucination, contradictions

The most important skills for a data analyst are not just running models but:
- Knowing when the results are trustworthy
- Validating systematically  
- Recognizing when human judgment is needed
- Documenting your process transparently

## Additional Resources

- [Google Gemini API Documentation](https://ai.google.dev/gemini-api/docs)
- [Structured Outputs Guide](https://ai.google.dev/gemini-api/docs/structured-output)
- [Pydantic Documentation](https://docs.pydantic.dev/)
- [Google's Best Practices for Prompt Engineering](https://ai.google.dev/gemini-api/docs/prompting-strategies)